# Retina U-Net Reproduction — CS598 DL4H

Runs the full reproduction end-to-end on Colab:

1. Clones this repo and checks out the `implementation1` branch.
2. Downloads a small slice of **MSD Task04 Hippocampus** (~27 MB, 3 patients, publicly hosted — no auth).
3. Visualizes a few slices with ground-truth bounding boxes.
4. Runs the **λ (`seg_weight`) ablation**, comparing Retina U-Net (λ > 0) against a **RetinaNet-only baseline** (λ = 0).
5. Plots per-epoch loss, validation detection F1 @ IoU 0.5, and binary segmentation IoU.

**Paper**: Jaeger et al., *Retina U-Net: Embarrassingly Simple Exploitation of Segmentation Supervision for Medical Object Detection*, [arXiv:1811.08661](https://arxiv.org/abs/1811.08661).

### Prerequisites
- Runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU). The sweep takes ~15–25 min on T4; ≥1 h on CPU.
- Branch `implementation1` must be **pushed to GitHub** before running the clone cell below. Edit `REPO_URL` to point to your fork if needed.


## 1. GPU check

In [ ]:
import torch

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"memory: {props.total_memory / 1e9:.1f} GB")
else:
    print("\u26a0\ufe0f  No GPU detected. Enable Runtime \u2192 Change runtime type \u2192 T4 GPU.")


## 2. Clone the repo and check out the implementation branch

In [ ]:
import os

REPO_URL = "https://github.com/jc182ill/CS598-HC.git"
BRANCH   = "implementation1"
REPO_DIR = "/content/CS598-HC"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH} || true
!git log -1 --oneline


## 3. Install the one extra dependency

Colab already has torch, torchvision, numpy, and matplotlib. We only need `nibabel` to read the NIfTI files from the Medical Segmentation Decathlon.

In [ ]:
!pip install -q nibabel


## 4. Download MSD Task04 Hippocampus and convert to our `.npy` layout

Downloads the ~27 MB tar from the public Medical Segmentation Decathlon S3 bucket, extracts 3 patients (imagesTr + labelsTr), and writes them into the layout the `RetinaUNetCTDataset` expects:

```
examples/data/hippocampus/
  patient_<id>/{volume,mask}.npy
```


In [ ]:
import os, subprocess, pathlib
import numpy as np
import nibabel as nib

DATA_TAR    = "/tmp/Task04_Hippocampus.tar"
EXTRACT_DIR = "/tmp"
DEST        = pathlib.Path("/content/CS598-HC/examples/data/hippocampus")
PATIENT_IDS = ["367", "304", "204"]

# Download (idempotent)
if not os.path.exists(DATA_TAR):
    !curl -L -o {DATA_TAR} https://msd-for-monai.s3-us-west-2.amazonaws.com/Task04_Hippocampus.tar

DEST.mkdir(parents=True, exist_ok=True)
for pid in PATIENT_IDS:
    pdir = DEST / f"patient_{pid}"
    if (pdir / "volume.npy").exists() and (pdir / "mask.npy").exists():
        print(f"patient_{pid}: already prepared, skipping")
        continue

    subprocess.run([
        "tar", "xf", DATA_TAR, "-C", EXTRACT_DIR,
        f"Task04_Hippocampus/imagesTr/hippocampus_{pid}.nii.gz",
        f"Task04_Hippocampus/labelsTr/hippocampus_{pid}.nii.gz",
    ], check=True)

    vol = nib.load(f"/tmp/Task04_Hippocampus/imagesTr/hippocampus_{pid}.nii.gz").get_fdata().astype(np.float32)
    lbl = nib.load(f"/tmp/Task04_Hippocampus/labelsTr/hippocampus_{pid}.nii.gz").get_fdata().astype(np.int32)

    # Clip top 0.5% outlier intensities (patient 204 has a stray spike).
    vol = np.clip(vol, 0, np.percentile(vol, 99.5))

    pdir.mkdir(exist_ok=True)
    np.save(pdir / "volume.npy", vol)
    np.save(pdir / "mask.npy", lbl)
    n_lesion = int((lbl.sum(axis=(1, 2)) > 0).sum())
    print(f"patient_{pid}: volume {vol.shape} float32, mask unique={np.unique(lbl).tolist()}, lesion slices={n_lesion}")

print(f"\nReady: {len(PATIENT_IDS)} patients at {DEST}")


## 5. Smoke test: dataset + task + visualization

Loads the 3 patients through `RetinaUNetCTDataset` + `RetinaUNetDetectionTask`, prints stats, and saves a figure with ground-truth boxes overlaid (red = anterior, blue = posterior hippocampus).

In [ ]:
%cd /content/CS598-HC
!PYTHONPATH=. python examples/hippocampus_retina_unet_demo.py

from IPython.display import Image, display
display(Image("/content/CS598-HC/examples/figures/hippocampus_demo.png"))


## 6. Unit tests (sanity, ~7 s on GPU)

Runs the full in-scope test suite (tasks + datasets + models + training helpers). These cover model instantiation, forward/backward on tiny tensors, loss arithmetic, and the detection/seg metric functions. Good canary before launching the longer ablation.


In [ ]:
!cd /content/CS598-HC && PYTHONPATH=. python -m pytest tests/tasks tests/datasets tests/models -q


## 7. λ ablation — RetinaNet baseline vs. Retina U-Net

Trains 4 fresh models with `seg_weight` ∈ {0, 0.5, 1.0, 2.0}. λ = 0 is the pure RetinaNet baseline (segmentation branch exists but doesn't contribute to the loss); λ > 0 is Retina U-Net with the segmentation supervision dialed in.

Patient split is deterministic: train on 2 patients, hold out 1 for validation. Metrics: detection F1 @ IoU 0.5, precision, recall, and binary segmentation IoU.

**Defaults**: 15 epochs, batch 8, Adam lr 1e-4. Adjust the env vars below for a quicker smoke test (e.g. `RUN_EPOCHS=3`).

In [ ]:
%%bash
cd /content/CS598-HC && PYTHONPATH=. \
  RUN_EPOCHS=15 \
  RUN_LAMBDAS=0,0.5,1.0,2.0 \
  RUN_BATCH_SIZE=8 \
  RUN_LR=1e-4 \
  RUN_SEED=42 \
  python examples/retina_unet_seg_weight_ablation.py


## 8. Final results: comparison table + plot

In [ ]:
import json
from IPython.display import Image, display

with open("/content/CS598-HC/examples/figures/ablation_results.json") as f:
    results = json.load(f)

print("=== Final val metrics (last epoch of each run) ===")
header = f"{'\u03bb':>5} | {'F1':>6} | {'precision':>9} | {'recall':>6} | {'seg IoU':>7}"
print(header)
print("-" * len(header))
for lam_str, hist in results["histories"].items():
    final = hist[-1]
    print(
        f"{float(lam_str):>5.2f} | "
        f"{final['val_f1']:>6.3f} | "
        f"{final['val_precision']:>9.3f} | "
        f"{final['val_recall']:>6.3f} | "
        f"{final['val_seg_iou']:>7.3f}"
    )

display(Image("/content/CS598-HC/examples/figures/ablation_results.png"))


## 9. Interpreting the results

According to the paper's central hypothesis, **segmentation supervision should improve detection** — that is, λ > 0 runs should achieve higher detection F1 than the λ = 0 RetinaNet baseline, especially with small training sets like this (~40 slices across 2 patients).

With only 15 epochs and 2 training patients the signal will be noisy; if you have GPU budget, push `RUN_EPOCHS` up and re-run. The JSON file in `examples/figures/ablation_results.json` has the full per-epoch history for deeper analysis.

**Known scope caveat**: this reproduces the paper's *method* on a small publicly-available MRI dataset, not the paper's *results* on LIDC-IDRI (which is CT, 125 GB, and auth-gated). See `PROJECT_STATUS.md` in the repo for the full scope vs. proposal comparison.
